In [14]:
import pandas as pd
import re

pop_new_raw = pd.read_csv("data/befolkningstal.csv", encoding="latin1", sep=";", header=None)
pop_old_raw = pd.read_csv("data/befolkningstalfør.csv", encoding="latin1", sep=";", header=None)

def clean_kommune_series(s):
    return (
        s.astype(str)
         .str.replace(r"\s*\(.*?\)", "", regex=True)
         .str.strip()
         .str.replace("Nykøbing-Falster", "Nykøbing Falster", regex=False)
         .str.replace("Lyngby-Tårbæk", "Lyngby-Taarbæk", regex=False)
    )

def clean_population_quarters_q1(df, start_year, end_year):
    """
    New population file:
    col 0 = køn
    col 1 = alder
    col 2 = civilstand
    col 3 = kommune
    col 4: = quarterly population values
    """
    df = df.copy()
    df = df.rename(columns={3: "kommune"})
    df["kommune"] = clean_kommune_series(df["kommune"])

    value_cols = list(df.columns[4:])
    records = []

    for i, col in enumerate(value_cols):
        year = start_year + i // 4
        quarter = (i % 4) + 1

        if year > end_year:
            break

        # Use K1/Q1 as annual population
        if quarter == 1:
            tmp = df[["kommune", col]].copy()
            tmp = tmp.rename(columns={col: "population"})
            tmp["år"] = year
            records.append(tmp)

    out = pd.concat(records, ignore_index=True)
    out["population"] = pd.to_numeric(out["population"], errors="coerce")

    return out[["kommune", "år", "population"]]

def clean_population_old_yearly(df, start_year, end_year):
    """
    Old population file:
    col 0 = køn
    col 1 = kommune
    col 2: = yearly population values
    """
    df = df.copy()
    df = df.rename(columns={1: "kommune"})
    df["kommune"] = clean_kommune_series(df["kommune"])

    years = list(range(start_year, end_year + 1))
    value_cols = list(df.columns[2:])

    if len(value_cols) != len(years):
        raise ValueError(
            f"Value columns: {len(value_cols)}, years: {len(years)}. "
            f"Check selected years or file structure."
        )

    df = df.rename(columns=dict(zip(value_cols, years)))

    out = df.melt(
        id_vars=["kommune"],
        value_vars=years,
        var_name="år",
        value_name="population"
    )

    out["år"] = out["år"].astype(int)
    out["population"] = pd.to_numeric(out["population"], errors="coerce")

    return out[["kommune", "år", "population"]]

# New data is quarterly, use K1/Q1 as annual population
pop_new_clean = clean_population_quarters_q1(pop_new_raw, 2007, 2025)

# Old data is yearly
pop_old_clean = clean_population_old_yearly(pop_old_raw, 1992, 2006)

print(pop_new_clean.head())
print(pop_new_clean["år"].min(), pop_new_clean["år"].max(), pop_new_clean["kommune"].nunique())

print(pop_old_clean.head())
print(pop_old_clean["år"].min(), pop_old_clean["år"].max(), pop_old_clean["kommune"].nunique())

         kommune    år  population
0      København  2007      509861
1  Frederiksberg  2007       93444
2         Dragør  2007       13261
3         Tårnby  2007       40016
4    Albertslund  2007       27602
2007 2025 99
                      kommune    år  population
0                 Hele landet  1992     5162126
1  København og Frederiksberg  1992      550938
2                   København  1992      464566
3               Frederiksberg  1992       86372
4              Københavns Amt  1992      603179
1992 2006 293


In [15]:
import pandas as pd

# ---------- Helper ----------
def clean_kommune_series(s):
    return (
        s.astype(str)
         .str.replace(r"\s*\(.*?\)", "", regex=True)
         .str.strip()
         .str.replace("Nykøbing-Falster", "Nykøbing Falster", regex=False)
         .str.replace("Lyngby-Tårbæk", "Lyngby-Taarbæk", regex=False)
    )

# ---------- Load mapping ----------
mapping = pd.read_excel(
    "data/Korrespondancetabel-mellem-kommuner-foer-og-efter-kommunalreformen-i-2007.xlsx"
)

mapping = mapping[["AMT_KOM_TXT", "NUTS_TXT"]].copy()
mapping.columns = ["old_kommune", "new_kommune"]

# Clean mapping names
mapping["old_kommune"] = clean_kommune_series(mapping["old_kommune"])
mapping["new_kommune"] = clean_kommune_series(mapping["new_kommune"])

# Fix special cases
mapping["new_kommune"] = mapping["new_kommune"].replace({
    "Bornholm excl. Christiansø": "Bornholm",
    "Christiansø Uden for Kommuner": None,
    "Ærø 2005/2006-": "Ærø"
})

mapping = mapping[mapping["new_kommune"].notna()]

# ---------- Manual fixes ----------
manual_mapping = pd.DataFrame({
    "old_kommune": [
        "Aakirkeby", "Allinge-Gudhjem", "Hasle",
        "Nexø", "Rønne", "Bornholm",
        "Ærøskøbing", "Marstal", "Ærø"
    ],
    "new_kommune": [
        "Bornholm", "Bornholm", "Bornholm",
        "Bornholm", "Bornholm", "Bornholm",
        "Ærø", "Ærø", "Ærø"
    ]
})

mapping = pd.concat([mapping, manual_mapping], ignore_index=True)
mapping = mapping.drop_duplicates(subset=["old_kommune"], keep="last")

# ---------- Clean old population ----------
pop_old_clean["kommune"] = clean_kommune_series(pop_old_clean["kommune"])

# ---------- Map old -> new ----------
pop_old_mapped = pop_old_clean.merge(
    mapping,
    left_on="kommune",
    right_on="old_kommune",
    how="left"
)

# Debug
missing = pop_old_mapped[
    pop_old_mapped["new_kommune"].isna()
]["kommune"].unique()

print("Missing population mappings:", missing)

# Drop unmapped (should only be Christiansø)
pop_old_mapped = pop_old_mapped[
    pop_old_mapped["new_kommune"].notna()
].copy()

# Replace kommune
pop_old_mapped["kommune"] = pop_old_mapped["new_kommune"]

# Aggregate to new municipalities
pop_old_mapped = pop_old_mapped.groupby(
    ["kommune", "år"], as_index=False
)["population"].sum()

# ---------- Clean new population ----------
pop_new_clean["kommune"] = clean_kommune_series(pop_new_clean["kommune"])

# Remove Christiansø if present
pop_new_clean = pop_new_clean[
    ~pop_new_clean["kommune"].str.contains("Christiansø", na=False)
].copy()

# ---------- Combine ----------
population = pd.concat(
    [
        pop_old_mapped[["kommune", "år", "population"]],
        pop_new_clean[["kommune", "år", "population"]]
    ],
    ignore_index=True
)

population = population.groupby(
    ["kommune", "år"], as_index=False
)["population"].sum()

population = population.sort_values(["kommune", "år"]).reset_index(drop=True)

# ---------- Checks ----------
print("Years:", population["år"].min(), "-", population["år"].max())
print("Municipalities:", population["kommune"].nunique())

population.head()

Missing population mappings: ['Hele landet' 'København og Frederiksberg' 'Københavns Amt'
 'Frederiksborg Amt' 'Roskilde Amt' 'Vestsjællands Amt' 'Storstrøms Amt'
 'Christiansø' 'Fyns Amt' 'Sønderjyllands Amt' 'Ribe Amt' 'Vejle Amt'
 'Ringkøbing Amt' 'Århus Amt' 'Viborg Amt' 'Nordjyllands Amt']
Years: 1992 - 2025
Municipalities: 98


,kommune,år,population
0,Aabenraa,1992,59903
1,Aabenraa,1993,59839
2,Aabenraa,1994,60036
3,Aabenraa,1995,59999
4,Aabenraa,1996,60045


In [16]:
population.to_csv("data/population_clean.csv", index=False)